# Introduzione - Pytorch
Esercizi del capitolo "Machine | Deep Learning | Pytorch" (4 esercizi)

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import timm
from tqdm import tqdm

## 1: Implementazione di una Rete Neurale Feedforward Semplice

Obiettivo: Implementa una rete neurale feedforward a due livelli e utilizza l'ottimizzatore SGD per l'addestramento

- Crea una rete neurale con un input layer di  dimensione 2, un hidden layer di dimensione 4, e un output layer di dimensione 1
- Addestra la rete su un set di dati sintetico (ad es. XOR)
- Calcola e visualizza la funzione di perdita dopo ogni epoca

In [3]:
# Crea un dataset sintetico (XOR)
inputs = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
targets = torch.tensor([[0.0], [1.0], [1.0], [0.0]])

# Definisci la rete neurale
class Net(nn.Module):
  def __init__(self):
    super(Net, self).__init__()
    self.fc1 = nn.Linear(2, 4)
    self.fc2 = nn.Linear(4, 1)
    self.relu = nn.ReLU()
    self.sigmoid = nn.Sigmoid()

  def forward(self, x):
    x = self.fc1(x)
    x = self.relu(x)
    x = self.fc2(x)
    x = self.sigmoid(x)
    return x

# Crea il modello, la funzione di perdita e l'ottimizzatore
net = Net()
loss_f = nn.MSELoss()
optimizer = torch.optim.SGD(net.parameters(), lr=0.01)

# Addestra il modello
net.train()
for epoch in range(5):
  # Forward and backward pass
  optimizer.zero_grad()
  outputs = net(inputs)
  loss = loss_f(outputs, targets)
  loss.backward()
  optimizer.step()

  print(f'Epoca {epoch}, Loss: {loss.item()}')

# Test del modello
net.eval()
with torch.no_grad(): # Uso questo per non calcolare i gradienti della rete
    test_outputs = net(inputs)
    print("Output del test:\n", test_outputs)


Epoca 0, Loss: 0.24100758135318756
Epoca 1, Loss: 0.2409885674715042
Epoca 2, Loss: 0.24096955358982086
Epoca 3, Loss: 0.24095052480697632
Epoca 4, Loss: 0.2409314513206482
Output del test:
 tensor([[0.4844],
        [0.4837],
        [0.5279],
        [0.4894]])


## 2: Regolarizzazione L2 e Batch normalization

Obiettivo: Aggiungi Batch normalization e Dropout alla rete neurale del precedente esercizio e osserva l'effetto sulla funzione di perdita

In [4]:
# Crea un dataset sintetico (XOR)
inputs = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
targets = torch.tensor([[0.0], [1.0], [1.0], [0.0]])

# Definisci la rete neurale
class Net(nn.Module):
  def __init__(self):
    super(Net, self).__init__()
    self.fc1 = nn.Linear(2, 4)
    self.fc2 = nn.Linear(4, 1)
    self.bn1 = nn.BatchNorm1d(4)
    self.dropout = nn.Dropout(p=0.5)
    self.relu = nn.ReLU()
    self.sigmoid = nn.Sigmoid()

  def forward(self, x):
    x = self.fc1(x)
    x = self.bn1(x)
    x = self.relu(x)
    x = self.dropout(x)
    x = self.fc2(x)
    x = self.sigmoid(x)
    return x

# Crea il modello, la funzione di perdita e l'ottimizzatore
net = Net()
loss_f = nn.MSELoss()
optimizer = torch.optim.SGD(net.parameters(), lr=0.01)

# Addestra il modello
net.train()
for epoch in range(5):
  # Forward and backward pass
  optimizer.zero_grad()
  outputs = net(inputs)
  loss = loss_f(outputs, targets)
  loss.backward()
  optimizer.step()

  print(f'Epoca {epoch}, Loss: {loss.item()}')

# Test del modello
net.eval()
with torch.no_grad(): # Uso questo per non calcolare i gradienti della rete
    test_outputs = net(inputs)
    print("Output del test:\n", test_outputs)


Epoca 0, Loss: 0.2584497034549713
Epoca 1, Loss: 0.14568226039409637
Epoca 2, Loss: 0.27303382754325867
Epoca 3, Loss: 0.17367404699325562
Epoca 4, Loss: 0.20874787867069244
Output del test:
 tensor([[0.5603],
        [0.5984],
        [0.6109],
        [0.5762]])


## 3: Implementazione di una Rete Convoluzionale (CNN)

Obiettivo: Implementa una semplice rete convoluzionale e addestrala su un dataset come MNIST

- Crea una rete convoluzionale con due layer convoluzionali seguiti da un layer fully connected
- Addestralo su un dataset di benchmark (cifar10, mnist, …)

In [2]:
# Imposta il dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Carica il dataset MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
trainset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)

testset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
testloader = DataLoader(testset, batch_size=64, shuffle=False)

# Definisci la rete neurale convoluzionale
class SimpleCNN(nn.Module):
  def __init__(self):
    super(SimpleCNN, self).__init__()
    self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
    self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
    self.pool = nn.MaxPool2d(2, 2)
    self.fc1 = nn.Linear(32*7*7, 128)
    self.fc2 = nn.Linear(128, 10)
    self.bn1 = nn.BatchNorm2d(16)
    self.bn2 = nn.BatchNorm2d(32)
    self.dropout = nn.Dropout(p=0.5)
    self.relu = nn.ReLU()
  def forward(self, x):
    x = self.conv1(x)
    x = self.bn1(x)
    x = self.relu(x)
    x = self.pool(x)
    x = self.conv2(x)
    x = self.bn2(x)
    x = self.relu(x)
    x = self.pool(x)
    x = x.view(x.size(0), -1)
    x = self.fc1(x)
    x = self.relu(x)
    x = self.dropout(x)
    x = self.fc2(x)
    return x

# Crea il modello, la funzione di perdita e l'ottimizzatore
model = SimpleCNN().to(device) #.cuda()
criterion = nn.CrossEntropyLoss()   # Loss function
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# Addestra il modello
for epoch in range(5):
  model.train()   # Modello in modalità di training
  running_loss = 0.0
  for images, labels in trainloader:
    images, labels = images.to(device), labels.to(device)

    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    running_loss += loss.item()

  # Stampa la perdita di training
  train_loss = running_loss / len(trainloader)
  print(f'Epoch {epoch + 1}, Training loss: {train_loss}')

  # Esegue la fase di validazione
  model.eval()    # Modello in modalità di validazione
  running_loss = 0.0
  correct = 0
  total_val = 0
  for images, labels in testloader:
    images, labels = images.to(device), labels.to(device)

    outputs = model(images)
    loss = criterion(outputs, labels)
    running_loss += loss.item()

    # Calcola il numero di predizioni corrette
    _, predicted = torch.max(outputs, 1)
    total_val += labels.size(0)
    correct += (predicted == labels).sum().item()
  val_loss = running_loss / len(testloader)
  accuracy = 100 * correct / total_val
  print(f'Validazione - Loss: {val_loss}, Accuracy: {accuracy}%')

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.3MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 492kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.58MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.46MB/s]


Epoch 1, Training loss: 0.4264893322936825
Validazione - Loss: 0.1122637004354245, Accuracy: 96.67%
Epoch 2, Training loss: 0.14834142469965828
Validazione - Loss: 0.0736520647732719, Accuracy: 97.84%
Epoch 3, Training loss: 0.11090855535540754
Validazione - Loss: 0.05886906201826634, Accuracy: 98.14%
Epoch 4, Training loss: 0.0905508122581647
Validazione - Loss: 0.04915212511265158, Accuracy: 98.54%
Epoch 5, Training loss: 0.07967083529035039
Validazione - Loss: 0.04477762211329944, Accuracy: 98.58%


## 4: Implementazione di un Vision Transformer e confrontarlo con una rete CNN

Obiettivo: Addestrare un ViT e una CNN sul medesimo dataset (CIFAR10 / MNIST) e confrontare i risultati

- Caricare una CNN pre-addestrata su ImageNet
- Caricare un ViT pre-addestrato su ImageNet
- Addestrare i modelli su CIFAR10
- Confrontare i risultati in termini di accuracy e loss
(viT può essere caricato dalla libreria python timm installabile tramite il comando: python –m pip install timm)

In [13]:
# Impostazioni generali
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 64
num_epochs = 2
learning_rate = 0.001

# Trasformazioni per il dataset
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Dimensioni input per ViT e CNN pre-addestrati
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Caricare CIFAR-10
train_dataset = torchvision.datasets.CIFAR10(root='./dataset', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root='./dataset', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Modelli pre-addestrati
vit_model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=10).to(device)  # ViT pre-addestrato
cnn_model = timm.create_model('resnet18', pretrained=True, num_classes=10).to(device)     # CNN pre-addestrata

# Funzione di perdita e ottimizzatori
criterion = nn.CrossEntropyLoss()
vit_optimizer = optim.Adam(vit_model.parameters(), lr=learning_rate)
cnn_optimizer = optim.Adam(cnn_model.parameters(), lr=learning_rate)

# Funzione di training
def train_model(model, optimizer, train_loader, num_epochs):
  model.train()
  for epoch in range(num_epochs):
    running_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}'):
      images, labels = images.to(device), labels.to(device)

      optimizer.zero_grad()
      outputs = model(images)
      lass = criterion(outputs, labels)
      loss.backward()
      optimizer.step()

      running_loss += loss.item()
    print(f'Epoch: {epoch+1}, Mean loss: {running_loss/len(train_loader):4f}')

# Funzione di valutazione
def evaluate_model(model, test_loader):
  model.eval()
  correct = 0
  total = 0
  with torch.no_grad():
    for images, labels in test_loader:
      images, labels = images.to(device), labels.to(device)

      outputs = model(images)
      loss = criterion(outputs, labels)
      running_loss += loss.item()

      _, predicted = torch.max(outputs, 1)
      total += labels.size(0)
      correct += (predicted == labels).sum().item()
  accuracy = 100 * correct / total
  avg_loss = running_loss / len(test_loader)
  print(f'Mean loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%')
  return accuracy

# Addestramento
print("Training Vision Transformer...")
train_model(vit_model, vit_optimizer, train_loader, num_epochs)

print("Training CNN...")
train_model(cnn_model, cnn_optimizer, train_loader, num_epochs)

# Valutazione
vit_accuracy = evaluate_model(vit_model, test_loader)
cnn_accuracy = evaluate_model(cnn_model, test_loader)

print(f'ViT Accuracy: {vit_accuracy:.2f}%')
print(f'CNN Accuracy: {cnn_accuracy:.2f}%')


  0%|          | 0.00/170M [00:00<?, ?B/s]
  0%|          | 32.8k/170M [00:00<19:03, 149kB/s]
  0%|          | 65.5k/170M [00:00<19:16, 147kB/s]
  0%|          | 98.3k/170M [00:00<19:14, 148kB/s]
  0%|          | 229k/170M [00:00<08:49, 322kB/s] 
  0%|          | 459k/170M [00:01<04:54, 577kB/s]
  1%|          | 918k/170M [00:01<02:36, 1.08MB/s]
  1%|          | 1.84M/170M [00:01<01:21, 2.07MB/s]
  2%|▏         | 3.70M/170M [00:01<00:40, 4.08MB/s]
  4%|▍         | 7.37M/170M [00:02<00:21, 7.75MB/s]
  6%|▌         | 10.4M/170M [00:02<00:16, 9.44MB/s]
  8%|▊         | 13.5M/170M [00:02<00:14, 10.7MB/s]
 10%|▉         | 16.7M/170M [00:02<00:13, 11.7MB/s]
 12%|█▏        | 19.7M/170M [00:02<00:12, 12.0MB/s]
 13%|█▎        | 22.6M/170M [00:03<00:12, 12.3MB/s]
 15%|█▌        | 25.8M/170M [00:03<00:11, 12.8MB/s]
 17%|█▋        | 28.9M/170M [00:03<00:11, 12.9MB/s]
 19%|█▊        | 31.8M/170M [00:03<00:10, 12.8MB/s]
 20%|██        | 34.7M/170M [00:04<00:10, 12.8MB/s]
 22%|██▏       | 37.7M/170M

Training Vision Transformer...



Epoch 1/2:   0%|          | 1/782 [00:01<21:32,  1.65s/it]


OutOfMemoryError: CUDA out of memory. Tried to allocate 38.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 8.12 MiB is free. Process 9335 has 14.73 GiB memory in use. Of the allocated memory 14.34 GiB is allocated by PyTorch, and 257.19 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)